# AAMem Phase Runner

Run the first cell on Colab. It clones or pulls the latest code from `https://github.com/lenhokhoa23/j4f.git`, then switches the runtime into that folder.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/lenhokhoa23/j4f.git'
COLAB_ROOT = Path('/content')

if COLAB_ROOT.exists():
    PROJECT_DIR = COLAB_ROOT / 'j4f'
    if (PROJECT_DIR / '.git').exists():
        subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR, check=True)
    elif not PROJECT_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
    else:
        print(f'{PROJECT_DIR} exists but is not a git repo; using current directory instead.')
        PROJECT_DIR = Path.cwd().resolve()
else:
    PROJECT_DIR = Path.cwd().resolve()

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('PROJECT_DIR =', PROJECT_DIR)


## Setup

Use `MODEL_PRESET='smoke'` for a no-model pipeline check. On Colab GPU, try `qwen2_5_1_5b`, then `qwen2_5_3b`, then `qwen2_5_7b`.

In [ ]:
from pathlib import Path
import json

MODEL_PRESET = 'smoke'
# MODEL_PRESET = 'qwen2_5_1_5b'
# MODEL_PRESET = 'qwen2_5_3b'
# MODEL_PRESET = 'qwen2_5_7b'

from aamem_lab.model_presets import presets_as_dict
presets_as_dict()


## Optional install for HF local models

Skip this for `MODEL_PRESET='smoke'`. Run it for Qwen/Llama/Gemma presets.

In [ ]:
# Uncomment on Colab GPU for HF model runs.
# %pip install -q transformers accelerate


## Official A-MEM robust baseline + Layer-2 gate

Ph?n n?y ch?y ??ng read-time pipeline robust c?a official A-MEM r?i so `none` v?i gate b?c ngo?i. Metric answer l?y t? official `utils.calculate_metrics`: exact match, F1, ROUGE-1/2/L, BLEU-1..4, BERTScore F1, METEOR, SBERT similarity. Runner s? print t?ng b??c ?? ??c live: question, retrieval query do A-MEM sinh, seed indices, s? candidate sau linked expansion, token context, label gate, prediction/gold v? metric.

In [ ]:
OFFICIAL_AMEM_URL = 'https://github.com/WujiangXu/AgenticMemory.git'
AMEM_CHECKOUT = PROJECT_DIR.parent / 'AgenticMemory'

if (AMEM_CHECKOUT / '.git').exists():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=AMEM_CHECKOUT, check=True)
elif not AMEM_CHECKOUT.exists():
    subprocess.run(['git', 'clone', OFFICIAL_AMEM_URL, str(AMEM_CHECKOUT)], check=True)
else:
    print(f'{AMEM_CHECKOUT} exists but is not a git repo; trying to use it as local A-MEM checkout.')

candidate_roots = [AMEM_CHECKOUT, AMEM_CHECKOUT / 'A-mem-main']
candidate_roots.extend([x for x in AMEM_CHECKOUT.glob('*') if x.is_dir()])
AMEM_REPO = next((x for x in candidate_roots if (x / 'test_advanced_robust.py').exists()), None)
if AMEM_REPO is None:
    raise FileNotFoundError(f'Cannot find official A-MEM root under {AMEM_CHECKOUT}')

AMEM_DATASET = AMEM_REPO / 'data' / 'locomo10.json'
print('AMEM_REPO =', AMEM_REPO)
print('AMEM_DATASET =', AMEM_DATASET, 'exists=', AMEM_DATASET.exists())


### Install official A-MEM dependencies

Ch?y cell n?y m?t l?n tr?n Colab. Official A-MEM c?n `sentence-transformers`, `rouge-score`, `bert-score`, `nltk`, `openai/ollama`, v.v. N?u runtime ?? c?i ??, c? th? ??i `INSTALL_OFFICIAL_AMEM_DEPS = False`.

In [ ]:
INSTALL_OFFICIAL_AMEM_DEPS = True
if INSTALL_OFFICIAL_AMEM_DEPS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(AMEM_REPO / 'requirements.txt')], check=True)
    import nltk
    for package in ['punkt', 'punkt_tab', 'wordnet', 'omw-1.4']:
        nltk.download(package, quiet=True)
    print('Official A-MEM deps and NLTK metric resources are ready.')


### Backend cho official A-MEM

M?c ??nh notebook d?ng `AMEM_BACKEND='hf'`, t?c l? g?i model HuggingFace tr?c ti?p b?ng `transformers`, kh?ng c?n vLLM server. ??y l? ???ng ?t r?i ro nh?t tr?n Colab khi vLLM wheel b? l?ch CUDA. N?u mu?n th? vLLM sau, ??i `AMEM_BACKEND='vllm'` v? b?t `START_VLLM_SERVER=True`.

In [ ]:
AMEM_BACKEND = 'hf'
AMEM_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
AMEM_GATE_BACKEND = AMEM_BACKEND
AMEM_GATE_MODEL = AMEM_MODEL

if AMEM_BACKEND == 'hf':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'accelerate'], check=True)
    print('Using local HuggingFace backend; no vLLM server is needed.')

VLLM_HOST = '127.0.0.1'
VLLM_PORT = 30000
VLLM_BASE_URL = f'http://{VLLM_HOST}:{VLLM_PORT}'
VLLM_LOG = PROJECT_DIR / 'runs' / 'vllm_qwen2_5_1_5b.log'
VLLM_PROCESS = None

def tail_file(path, n=80):
    path = Path(path)
    if not path.exists():
        return f'Log file does not exist yet: {path}'
    lines = path.read_text(encoding='utf-8', errors='replace').splitlines()
    return '\n'.join(lines[-n:])

def is_vllm_ready(timeout=5):
    if AMEM_BACKEND != 'vllm':
        return True
    import urllib.request
    url = f'{VLLM_BASE_URL}/v1/models'
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            return response.status == 200
    except Exception:
        return False

def wait_for_amem_backend_ready(max_wait_sec=900, poll_sec=10, process=None, log_path=None):
    if AMEM_BACKEND != 'vllm':
        print(f'No HTTP preflight for backend={AMEM_BACKEND}; runner will validate it on first LLM call.')
        return
    import time
    import urllib.request
    url = f'{VLLM_BASE_URL}/v1/models'
    deadline = time.time() + max_wait_sec
    attempt = 0
    while True:
        attempt += 1
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                print('vLLM preflight OK:', response.status, url)
                return
        except Exception as exc:
            if process is not None and process.poll() is not None:
                print('\n=== vLLM log tail ===')
                print(tail_file(log_path or VLLM_LOG, 120))
                raise RuntimeError(f'vLLM process exited with code {process.returncode}. See log above.') from exc
            if time.time() >= deadline:
                print('\n=== vLLM log tail ===')
                print(tail_file(log_path or VLLM_LOG, 120))
                raise RuntimeError(f'vLLM backend was not ready after {max_wait_sec}s at {url}. See log above.') from exc
            print(f'[vLLM wait] attempt={attempt} not ready yet: {type(exc).__name__}: {exc}')
            if attempt % 6 == 0:
                print('\n=== vLLM log tail ===')
                print(tail_file(log_path or VLLM_LOG, 40))
            time.sleep(poll_sec)

def assert_amem_backend_ready():
    wait_for_amem_backend_ready(
        max_wait_sec=900,
        poll_sec=10,
        process=VLLM_PROCESS,
        log_path=VLLM_LOG,
    )

START_VLLM_SERVER = False
if START_VLLM_SERVER:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'], check=True)
    VLLM_LOG.parent.mkdir(parents=True, exist_ok=True)
    if is_vllm_ready(timeout=2):
        print('Existing vLLM server is already ready:', VLLM_BASE_URL)
    else:
        log_handle = VLLM_LOG.open('w', encoding='utf-8')
        VLLM_PROCESS = subprocess.Popen([
            sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
            '--model', AMEM_MODEL,
            '--host', '0.0.0.0',
            '--port', str(VLLM_PORT),
            '--max-model-len', '8192',
            '--gpu-memory-utilization', '0.90',
            '--trust-remote-code',
        ], stdout=log_handle, stderr=subprocess.STDOUT)
        print('vLLM PID =', VLLM_PROCESS.pid)
        print('vLLM log =', VLLM_LOG)
    wait_for_amem_backend_ready(max_wait_sec=900, poll_sec=10, process=VLLM_PROCESS, log_path=VLLM_LOG)
elif AMEM_BACKEND == 'vllm':
    print('Backend ch?a ???c t? b?t. N?u ch?a c? vLLM/Ollama/SGLang/OpenAI s?n, b?t START_VLLM_SERVER=True r?i ch?y l?i cell n?y.')
else:
    print(f'Using backend={AMEM_BACKEND}; no local server startup in this cell.')


### Run official A-MEM: baseline kh?ng b?c vs gate c?ch 1/c?ch 2

M?c ??nh ch?y `none,heuristic` ?? so baseline official v?i c?ch 1 tr??c. Khi backend ?n v? mu?n ch?y c?ch 2, ??i `AMEM_GATES='none,heuristic,llm'`. `AMEM_MAX_QUESTIONS=30` l? subset to h?n smoke test nh?ng v?n c?n v?a ?? debug; t?ng l?n `60`, `100`, ho?c d?ng `AMEM_RATIO=0.3/0.5/1.0` khi ?? ?n.

In [ ]:
AMEM_GATES = 'none,heuristic'
# AMEM_GATES = 'none,heuristic,llm'
AMEM_MAX_QUESTIONS = 30
AMEM_RATIO = 1.0
AMEM_RETRIEVE_K = 10
AMEM_TAG = f'official_locomo_n{AMEM_MAX_QUESTIONS}_k{AMEM_RETRIEVE_K}_{AMEM_GATES.replace(",", "_")}_{AMEM_MODEL.split("/")[-1]}'
OFFICIAL_OUT_DIR = PROJECT_DIR / 'runs' / 'official_amem_gate'
OFFICIAL_SUMMARY = OFFICIAL_OUT_DIR / f'{AMEM_TAG}_summary.json'

print(f'AMEM_RATIO={AMEM_RATIO}: fraction of LoCoMo conversation samples loaded before the question cap.')
print(f'AMEM_MAX_QUESTIONS={AMEM_MAX_QUESTIONS}: hard cap on evaluated QA items; this is the main quick-run size control.')
assert_amem_backend_ready()

cmd = [
    sys.executable, '-u', '-m', 'aamem_lab.official_amem_gate_runner',
    '--amem-repo', str(AMEM_REPO),
    '--dataset', str(AMEM_DATASET),
    '--backend', AMEM_BACKEND,
    '--model', AMEM_MODEL,
    '--gate-backend', AMEM_GATE_BACKEND,
    '--gate-model', AMEM_GATE_MODEL,
    '--hf-max-new-tokens', '768',
    '--gates', AMEM_GATES,
    '--ratio', str(AMEM_RATIO),
    '--max-questions', str(AMEM_MAX_QUESTIONS),
    '--retrieve-k', str(AMEM_RETRIEVE_K),
    '--include-prompts',
    '--include-contexts',
    '--output-dir', str(OFFICIAL_OUT_DIR),
    '--tag', AMEM_TAG,
]
def run_streaming_command(cmd, cwd, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('Running command:')
    print(' '.join(map(str, cmd)))
    print('Run log =', log_path)
    with log_path.open('w', encoding='utf-8') as log_f:
        proc = subprocess.Popen(
            cmd,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_f.write(line)
            log_f.flush()
        return_code = proc.wait()
    if return_code != 0:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print('\n=== official A-MEM run log tail ===')
        print('\n'.join(lines[-120:]))
        raise RuntimeError(f'Official A-MEM run failed with exit code {return_code}. See {log_path}')

RUN_LOG = OFFICIAL_OUT_DIR / f'{AMEM_TAG}.run.log'
run_streaming_command(cmd, cwd=PROJECT_DIR, log_path=RUN_LOG)
print('OFFICIAL_SUMMARY =', OFFICIAL_SUMMARY)


### B?ng so s?nh cu?i

Cell n?y ??c summary v?a sinh v? in b?ng metric official c?a A-MEM theo t?ng gate, k?m b?ng token/context ?? xem gate c? gi?m prompt hay l?m t?t/t?ng quality kh?ng.

In [ ]:
subprocess.run([
    sys.executable, 'scripts/print_official_amem_gate_summary.py',
    str(OFFICIAL_SUMMARY),
    '--show-examples', '3',
], cwd=PROJECT_DIR, check=True)


## Phase 2: ActMem answer-level run

Outputs go to `runs/notebook_phase2`. The JSONL contains context, answer, judge, paper metrics, and run config.

In [ ]:
cmd = [
    sys.executable, '-m', 'aamem_lab.phase2_answer_runner',
    '--dataset', 'actmem',
    '--limit', '10',
    '--k', '5',
    '--model-preset', MODEL_PRESET,
    '--include-prompts',
    '--out-dir', str(PROJECT_DIR / 'runs' / 'notebook_phase2'),
    '--tag', f'actmem_n10_k5_{MODEL_PRESET}',
]
subprocess.run(cmd, cwd=PROJECT_DIR, check=True)


## Phase 3: STALE-style SR/PR/IPA run

This is our synthetic stale suite aligned to status recognition, premise resistance, and implicit preference/action.

In [ ]:
cmd = [
    sys.executable, '-m', 'aamem_lab.phase3_stale_runner',
    '--limit', '12',
    '--k', '5',
    '--dimensions', 'SR,PR,IPA',
    '--model-preset', MODEL_PRESET,
    '--include-prompts',
    '--out-dir', str(PROJECT_DIR / 'runs' / 'notebook_phase3'),
    '--tag', f'stale_n12_k5_sr_pr_ipa_{MODEL_PRESET}',
]
subprocess.run(cmd, cwd=PROJECT_DIR, check=True)


## Read summaries

In [ ]:
for path in sorted((PROJECT_DIR / 'runs').glob('notebook_*/*_summary.json')):
    print('\n===', path, '===')
    print(path.read_text(encoding='utf-8')[:4000])
